In [ ]:
import json
from app import app
from database import db, POI
import pandas as pd
from sqlalchemy import text
from tqdm import tqdm

In [14]:
data = json.load(open("data/poi_metadata.json"))

In [15]:
df = pd.DataFrame.from_dict(data, orient="index")

In [16]:
df.columns

Index(['raw_poi_id', 'latitude', 'longitude', 'checkins_count_from_events',
       'users_count_from_events', 'item_id', 'spot_latitude', 'spot_longitude',
       'category_id', 'category_name', 'raw_categories', 'photos_count',
       'checkins_count', 'users_count', 'radius_meters', 'highlights_count',
       'items_count', 'max_items_count', 'created_at'],
      dtype='str')

In [17]:
df.head()

,raw_poi_id,latitude,longitude,checkins_count_from_events,users_count_from_events,item_id,spot_latitude,spot_longitude,category_id,category_name,raw_categories,photos_count,checkins_count,users_count,radius_meters,highlights_count,items_count,max_items_count,created_at
8938,8938,39.052824,-94.590311,68.0,18.0,1089.0,39.052824,-94.590311,1.0,Coffee Shop,"[{'url': '/categories/1', 'name': 'Coffee Shop'}]",38.0,438.0,94.0,50.0,10.0,10.0,10.0,2008-12-15T00:22:49Z
8947,8947,37.331880,-122.029631,403.0,199.0,375.0,37.331880,-122.029631,121.0,Corporate Office,"[{'url': '/categories/121', 'name': 'Corporate...",91.0,3100.0,1186.0,200.0,20.0,10.0,10.0,2008-12-16T23:14:05Z
8956,8956,32.942655,-97.131200,40.0,28.0,2687.0,32.942655,-97.131200,125.0,City Hall,"[{'url': '/categories/125', 'name': 'City Hall'}]",5.0,335.0,146.0,75.0,0.0,10.0,10.0,2008-12-18T23:38:58Z
8957,8957,32.942222,-97.131213,22.0,16.0,2686.0,32.942222,-97.131213,150.0,Fountain,"[{'url': '/categories/150', 'name': 'Fountain'}]",8.0,196.0,101.0,50.0,0.0,10.0,10.0,2008-12-18T23:42:09Z
8958,8958,32.882931,-97.260221,26.0,20.0,4029.0,32.882931,-97.260221,363.0,Chipotle,"[{'url': '/categories/363', 'name': 'Chipotle'}]",9.0,221.0,102.0,75.0,0.0,10.0,10.0,2008-12-19T00:05:52Z


In [18]:
df.shape

(2725573, 19)

In [36]:
for col in df.columns:
    if col != "raw_categories":
        print(col, df[col].nunique())

raw_poi_id 2725573
latitude 2630109
longitude 2629489
checkins_count_from_events 447
users_count_from_events 341
item_id 29511
spot_latitude 2629551
spot_longitude 2628968
category_id 667
category_name 630
photos_count 283
checkins_count 1832
users_count 1272
radius_meters 69
highlights_count 80
items_count 24
max_items_count 1
created_at 2641598


In [19]:
df.isnull().sum()

raw_poi_id                          0
latitude                            0
longitude                           0
checkins_count_from_events    2695878
users_count_from_events       2695878
item_id                       2696062
spot_latitude                     682
spot_longitude                    682
category_id                       682
category_name                     682
raw_categories                    682
photos_count                      682
checkins_count                    682
users_count                       682
radius_meters                     682
highlights_count                  682
items_count                       682
max_items_count                   682
created_at                        682
dtype: int64

In [20]:
df_filtered = df[~df["item_id"].isna()]

In [21]:
df_filtered.shape

(29511, 19)

In [22]:
df_filtered.isnull().sum()

raw_poi_id                      0
latitude                        0
longitude                       0
checkins_count_from_events      0
users_count_from_events         0
item_id                         0
spot_latitude                 662
spot_longitude                662
category_id                   662
category_name                 662
raw_categories                662
photos_count                  662
checkins_count                662
users_count                   662
radius_meters                 662
highlights_count              662
items_count                   662
max_items_count               662
created_at                    662
dtype: int64

In [23]:
df_filtered.to_json("data/filtered_poi_metadata.json", orient="index")

In [ ]:
data = json.load(open("data/filtered_poi_metadata.json"))

In [41]:
df2 = pd.DataFrame.from_dict(data, orient="index")

In [44]:
df2.shape

(29511, 19)

In [48]:
with app.app_context():
    db.drop_all()
    db.create_all()

    db.session.execute(text("PRAGMA synchronous = OFF;"))
    db.session.execute(text("PRAGMA journal_mode = MEMORY;"))
    db.session.execute(text("PRAGMA cache_size = -1000000;"))

    pois_to_insert = []
    BATCH_SIZE = 10000
    for poi_id, poi_data in tqdm(data.items(), total=len(data)):
        # print(poi_id, poi_data)
        poi_dict = {
            "id": int(poi_id),
            "item_id": poi_data.get("item_id"),
            "lat": poi_data.get("latitude"),
            "lng": poi_data.get("longitude"),
            "cat_name": poi_data.get("category_name"),
            "raw_data": poi_data,
        }

        pois_to_insert.append(poi_dict)

        if len(pois_to_insert) >= BATCH_SIZE:
            db.session.bulk_insert_mappings(POI, pois_to_insert)
            db.session.commit()
            pois_to_insert.clear()

    if pois_to_insert:
        db.session.bulk_insert_mappings(POI, pois_to_insert)
        db.session.commit()

100%|██████████| 29511/29511 [00:00<00:00, 45111.06it/s]


In [52]:
with app.app_context():
    print(db.session.query(POI).count())

29511
